# ERP Tabanlı Müşteri Analitiği Projesi

Bu projede müşteri verileri analiz edilerek RFM segmentasyonu, churn analizi ve Pareto analizi gerçekleştirilmiştir.

In [6]:
import pandas as pd
import numpy as np
import sqlite3

In [7]:
df = pd.read_csv("flo_data_20k (1).csv")


In [8]:
df.head(5)

,master_id,order_channel,last_order_channel,first_order_date,last_order_date,last_order_date_online,last_order_date_offline,order_num_total_ever_online,order_num_total_ever_offline,customer_value_total_ever_offline,customer_value_total_ever_online,interested_in_categories_12
0,cc294636-19f0-11eb-8d74-000d3a38a36f,Android App,Offline,2020-10-30,2021-02-26,2021-02-21,2021-02-26,4.0,1.0,139.99,799.38,[KADIN]
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,Android App,Mobile,2017-02-08,2021-02-16,2021-02-16,2020-01-10,19.0,2.0,159.97,1853.58,"[ERKEK, COCUK, KADIN, AKTIFSPOR]"
2,69b69676-1a40-11ea-941b-000d3a38a36f,Android App,Android App,2019-11-27,2020-11-27,2020-11-27,2019-12-01,3.0,2.0,189.97,395.35,"[ERKEK, KADIN]"
3,1854e56c-491f-11eb-806e-000d3a38a36f,Android App,Android App,2021-01-06,2021-01-17,2021-01-17,2021-01-06,1.0,1.0,39.99,81.98,"[AKTIFCOCUK, COCUK]"
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,Desktop,Desktop,2019-08-03,2021-03-07,2021-03-07,2019-08-03,1.0,1.0,49.99,159.99,[AKTIFSPOR]


In [9]:
df.shape

(19945, 12)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19945 entries, 0 to 19944
Data columns (total 12 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   master_id                          19945 non-null  object 
 1   order_channel                      19945 non-null  object 
 2   last_order_channel                 19945 non-null  object 
 3   first_order_date                   19945 non-null  object 
 4   last_order_date                    19945 non-null  object 
 5   last_order_date_online             19945 non-null  object 
 6   last_order_date_offline            19945 non-null  object 
 7   order_num_total_ever_online        19945 non-null  float64
 8   order_num_total_ever_offline       19945 non-null  float64
 9   customer_value_total_ever_offline  19945 non-null  float64
 10  customer_value_total_ever_online   19945 non-null  float64
 11  interested_in_categories_12        19945 non-null  obj

In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
order_num_total_ever_online,19945.0,3.110855,4.225647,1.00,1.00,2.00,4.00,200.00
order_num_total_ever_offline,19945.0,1.913913,2.062880,1.00,1.00,1.00,2.00,109.00
customer_value_total_ever_offline,19945.0,253.922597,301.532853,10.00,99.99,179.98,319.97,18119.14
customer_value_total_ever_online,19945.0,497.321690,832.601886,12.99,149.98,286.46,578.44,45220.13


In [12]:
df.isnull().sum()

,0
master_id,0
order_channel,0
last_order_channel,0
first_order_date,0
last_order_date,0
last_order_date_online,0
last_order_date_offline,0
order_num_total_ever_online,0
order_num_total_ever_offline,0
customer_value_total_ever_offline,0


In [13]:
df.duplicated().sum()

np.int64(0)

In [48]:
df["interested_in_categories_12"].value_counts().head(5)

,count
interested_in_categories_12,
[AKTIFSPOR],3464
[KADIN],2158
[],2135
[ERKEK],1973
"[KADIN, AKTIFSPOR]",1352


In [14]:
# Toplam müşteri sayısı
df["master_id"].nunique()


19945

In [50]:
df[["order_num_total_ever_online", "order_num_total_ever_offline"]].mean()

,0
order_num_total_ever_online,3.110855
order_num_total_ever_offline,1.913913


In [51]:
df[["customer_value_total_ever_online", "customer_value_total_ever_offline"]].mean()

,0
customer_value_total_ever_online,497.321690
customer_value_total_ever_offline,253.922597


In [25]:
df["total_order_num"] = df["order_num_total_ever_online"] + df["order_num_total_ever_offline"] #Online ve offline satışları topladık.

df["total_customer_value"] = (df["customer_value_total_ever_online"] + df["customer_value_total_ever_offline"]) #Online ve offline toplam harcamalar.

df["avg_order_value"] = df["total_customer_value"] / df["total_order_num"] #Bu ikisinin bölerek ortalama siparişi hesapladım.


In [19]:
# Toplam gelir
df["total_customer_value"].sum()



np.float64(14983567.309999999)

In [26]:
df["first_order_date_dt"] = pd.to_datetime(df["first_order_date"])
df["last_order_date_dt"] = pd.to_datetime(df["last_order_date"])
# Burada firstorderdate sütunundaki değerler string'ti onu doğru forma soktum.

In [28]:
df[["first_order_date_dt","last_order_date_dt"]].head()

,first_order_date_dt,last_order_date_dt
0,2020-10-30,2021-02-26
1,2017-02-08,2021-02-16
2,2019-11-27,2020-11-27
3,2021-01-06,2021-01-17
4,2019-08-03,2021-03-07


In [21]:
df["avg_order_value"]

,avg_order_value
0,187.874000
1,95.883333
2,117.064000
3,60.985000
4,104.990000
...,...
19940,133.986667
19941,195.235000
19942,210.980000
19943,168.295000


In [52]:
df["avg_order_value"].describe().round(0)

,avg_order_value
count,19945.0
mean,152.0
std,84.0
min,22.0
25%,103.0
50%,137.0
75%,182.0
max,5177.0


In [20]:
df[["total_order_num", "total_customer_value", "avg_order_value"]].describe().T
# Müşterilerin sipariş sayısı, toplam harcaması ve ortalama sipariş değeri incelenmiştir.
# Ortalama sipariş sayısı 5'dir. Sapmanın bu kadar yüksek olması tüketici profili olarak geniş bir kitle olduğunu gösterir.

,count,mean,std,min,25%,50%,75%,max
total_order_num,19945.0,5.024768,4.742707,2.00,3.00,4.000,6.000,202.000
total_customer_value,19945.0,751.244287,895.402173,44.98,339.98,545.270,897.780,45905.100
avg_order_value,19945.0,152.399446,83.503935,22.49,103.49,136.735,182.445,5176.585


In [29]:
df["customer_life"] = (df["last_order_date_dt"] - df["first_order_date_dt"]).dt.days.astype("Int64")

In [63]:
df[["last_order_date_dt","first_order_date_dt","customer_life"]].sort_values(by="customer_life", ascending=False).head()
# Burada en uzun süreli ilk 5 müşteri ve tarihleri verilmiştir.

,last_order_date_dt,first_order_date_dt,customer_life
15271,2021-05-27,2013-02-04,3034
7629,2021-05-30,2013-02-10,3031
12967,2021-05-06,2013-01-20,3028
3568,2021-05-17,2013-02-10,3018
17616,2021-04-18,2013-01-14,3016


In [31]:
df.loc[df["customer_life"].idxmax()] #Burda ise en uzun süredir alışveriş yapan kullanıcının istatistikleri yer alıyor.

,15271
master_id,00cf8494-9da2-11e9-9897-000d3a38a36f
order_channel,Ios App
last_order_channel,Desktop
first_order_date,2013-02-04
last_order_date,2021-05-27
last_order_date_online,2021-05-27
last_order_date_offline,2020-08-28
order_num_total_ever_online,48.0
order_num_total_ever_offline,5.0
customer_value_total_ever_offline,296.94


In [41]:
df.nlargest(5, "total_customer_value")[["master_id","total_customer_value"]]
#En yüksek harcama yapan müşterileri inceleme

,master_id,total_customer_value
11150,5d1c466a-9cfd-11e9-9897-000d3a38a36f,45905.10
4315,d5ef8058-a5c6-11e9-a2fc-000d3a38a36f,36818.29
7613,73fd19aa-9e37-11e9-9897-000d3a38a36f,33918.10
13880,7137a5c0-7aad-11ea-8f20-000d3a38a36f,31227.41
9055,47a642fe-975b-11eb-8c2a-000d3a38a36f,20706.34


In [43]:
df.nlargest(5, "total_order_num")[["master_id","total_order_num"]]

,master_id,total_order_num
11150,5d1c466a-9cfd-11e9-9897-000d3a38a36f,202.0
7223,cba59206-9dd1-11e9-9897-000d3a38a36f,131.0
8783,a57f4302-b1a8-11e9-89fa-000d3a38a36f,111.0
2619,fdbe8304-a7ab-11e9-a2fc-000d3a38a36f,88.0
6322,329968c6-a0e2-11e9-a2fc-000d3a38a36f,83.0


In [44]:
df.groupby("order_channel")["total_customer_value"].sum().sort_values(ascending=False)
#Android platformu toplam gelirde lider konumda iken;

,total_customer_value
order_channel,
Android App,7819062.76
Mobile,3028183.16
Ios App,2525999.93
Desktop,1610321.46


In [46]:
df.groupby("order_channel")["total_customer_value"].mean().sort_values(ascending=False)
#iOS kullanıcıları müşteri başına daha yüksek harcama yaparak daha değerli bir segment oluşturmaktadır.

,total_customer_value
order_channel,
Ios App,891.634285
Android App,823.492655
Mobile,620.275125
Desktop,588.782984


In [64]:
# Kanal bazlı müşteri sayısı
df.groupby("order_channel")["master_id"].nunique()

,master_id
order_channel,
Android App,9495
Desktop,2735
Ios App,2833
Mobile,4882


In [45]:
channel_sales = df.groupby("order_channel")["total_customer_value"].sum().sort_values(ascending=False)

(channel_sales / channel_sales.sum()) * 100

,total_customer_value
order_channel,
Android App,52.184254
Mobile,20.210028
Ios App,16.858468
Desktop,10.747250


In [32]:
today_date = df["last_order_date_dt"].max() + pd.Timedelta(days=1)

df["recency"] = (today_date - df["last_order_date_dt"]).dt.days.astype("Int64")
# Recency değişkeni hesaplanarak müşterinin son alışverişinden bu yana geçen gün sayısı bulunmuştur.
# Bu metrik müşterinin ne kadar aktif olduğunu ve churn yani kaybedilme riskini analiz etmek için kullanılır.

In [34]:
rfm = df.groupby("master_id").agg({
    "recency": "min",
    "total_order_num": "sum",
    "total_customer_value": "sum"
})

rfm.columns = ["recency", "frequency", "monetary"]
# Müşteriler özelinde teker teker rfm tablosu yapılmıştır.

In [35]:
rfm["R_score"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])
rfm["M_score"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5])
# Rfm metrikleri 5 e kadar sıralandı.
#Recencynin düşük olmasının sebebi recency ters çalışır.Düşük olan yani yakın zamanda alışveriş yapan müşterilere yüksek skor 5 verilmiştir.
# Frequency ve Monetary değerleri yüksek olan müşterilere ise daha yüksek skorlar atanmıştır.
# Bu skorlar sayesinde müşteriler davranışlarına göre karşılaştırılabilir hale getirilmiştir.

In [36]:
rfm["RFM_SCORE"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str)
# Monetary neden dışarıda derseniz bazı müşteriler az alışveriş yapar ama pahalı ürün alır
#bazıları çok alışveriş yapar ama ucuz bu segmentasyonu bozabilir.

In [37]:
seg_map = {
    r"[1-2][1-2]": "hibernating",
    r"[1-2][3-4]": "at_risk",
    r"[1-2]5": "cant_lose",
    r"3[1-2]": "about_to_sleep",
    r"33": "need_attention",
    r"[3-4][4-5]": "loyal_customers",
    r"41": "promising",
    r"51": "new_customers",
    r"[4-5][2-3]": "potential_loyalists",
    r"5[4-5]": "champions"
}

rfm["segment"] = rfm["RFM_SCORE"].replace(seg_map, regex=True)
# Bu segmentler sayesinde müşterilerin davranışları anlamlandırılmıştır
# Örneğin 55 champions en iyi müşteri 11 hibernating pasif müşteri 15 cant_lose kaybedilmemeli gibi
# Böylece her müşteri grubuna özel pazarlama ve iş stratejileri geliştirilebilir.

In [38]:
rfm["segment"].value_counts() #Burada da yukarıda yaptığımızın sonuçları.
# En büyük grubun hibernating olduğu görülüyor.
# Bu durum müşteri kitlesinin önemli bir kısmının uzun süredir aktif olmadığını ve yeniden kazanım stratejilerine ihtiyaç duyulduğunu gösteriyor.
# Loyal_customers ve champions segmentlerinin de yüksek sayıda olması, mevcut aktif ve değerli müşteri kitlesinin güçlü olduğunu gösteriyor.
# At_risk ve cant_lose segmentleri ise geçmişte değerli olup şu an kaybedilme riski taşıyan müşterileri ifade ediyor.


,count
segment,
hibernating,3589
loyal_customers,3375
at_risk,3152
potential_loyalists,2925
champions,1920
about_to_sleep,1643
cant_lose,1194
need_attention,806
new_customers,673


In [39]:
rfm.groupby("segment").agg({
    "recency": "mean",
    "frequency": "mean",
    "monetary": "mean"
    }).sort_values("monetary", ascending=False)
# Bu sonuçlara göre cant_lose ve champions segmentlerinin en yüksek harcama (monetary) değerine sahip olduğu görülmektedir.
# Champions segmenti düşük recency değeri ile aktif ve yüksek frekanslı alışveriş yapan en değerli müşteri grubunu temsil etmektedir.
# Cant_lose segmenti ise yüksek harcama ve frekansa sahip olmasına rağmen yüksek recency değeri ile uzun süredir alışveriş yapmayan kritik müşterileri göstermektedir.
# Loyal_customers segmenti düzenli alışveriş yapan ve istikrarlı gelir sağlayan müşteri grubudur.
# Hibernating ve "about_to_sleep" segmentleri düşük frekans ve düşük harcama değerleri ile pasif müşteri kitlesini temsil etmektedir.
# Bu analiz, hangi müşteri segmentine nasıl aksiyon alınması gerektiğini belirlemek açısından önemli bir yol gösteriyor.

,recency,frequency,monetary
segment,,,
cant_lose,234.159129,10.716918,1481.652446
champions,16.142187,8.965104,1410.708938
loyal_customers,81.557926,8.356444,1216.257224
at_risk,241.328997,4.470178,648.325038
need_attention,112.037221,3.739454,553.436638
potential_loyalists,35.869744,3.310769,533.741344
hibernating,246.426303,2.391474,362.583299
about_to_sleep,113.031649,2.406573,361.649373
new_customers,16.976226,2.000000,344.049495


In [42]:
df.nlargest(5, "recency")[["master_id","recency"]]

,master_id,recency
482,cf003ff8-ddcf-11e9-a848-000d3a38a36f,366
2225,043f4824-a5d3-11e9-a2fc-000d3a38a36f,366
2310,62a7fbcc-aa1a-11e9-a2fc-000d3a38a36f,366
3161,32314cda-b1ba-11e9-89fa-000d3a38a36f,366
3205,929228ac-a5d4-11e9-a2fc-000d3a38a36f,366


In [53]:
df["churn"] = df["recency"] > 90 # Yani 90 günden beri yoksa müşteri kaybedilmiş churn olarak adlandırılıcak.

In [54]:
df["churn"].value_counts()
#Sonuçlar incelendiğinde churn olan müşteri sayısının,
#Churn olmayanlardan daha fazla olduğu görülmektedir.

,count
churn,
True,11329
False,8616


In [55]:
df["churn"].mean() * 100

np.float64(56.801203309100025)

In [56]:
df.groupby("churn").agg({
    "total_customer_value": "mean",
    "total_order_num": "mean",
    "customer_life": "mean"
})
# Churn olan ve olmayan müşteriler karşılaştırıldığında, aktif müşterilerin (churn=False) daha yüksek ortalama harcama yaptığı görülmektedir.
# Ayrıca aktif müşterilerin ortalama sipariş sayısının daha fazla olduğu, yani daha sık alışveriş yaptıkları anlaşılmaktadır.
# Müşteri yaşam süresi (customer_life) incelendiğinde ise aktif müşterilerin daha uzun süre sistemde kaldığı görülmektedir.
# Churn olan müşteriler hem daha az alışveriş yapmakta hem de daha düşük gelir sağlamaktadır.


,total_customer_value,total_order_num,customer_life
churn,,,
False,868.063174,5.593895,711.321727
True,662.400477,4.591932,633.01836


In [57]:
pareto = df.sort_values("total_customer_value", ascending=False)

In [58]:
pareto["cum_value"] = pareto["total_customer_value"].cumsum()


In [59]:
pareto["cum_ratio"] = pareto["cum_value"] / pareto["total_customer_value"].sum()

In [60]:
pareto["customer_ratio"] = np.arange(1, len(pareto)+1) / len(pareto)

In [61]:
pareto[pareto["cum_ratio"] <= 0.80].tail()
#Gelirin %80’inin %53 müşteri tarafından oluşturulduğu müşteri değerinin geniş bir kitleye yayıldığını ve yalnızca dar bir VIP segmentine bağlı olmadığını göstermektedir.

,master_id,order_channel,last_order_channel,first_order_date,last_order_date,last_order_date_online,last_order_date_offline,order_num_total_ever_online,order_num_total_ever_offline,customer_value_total_ever_offline,...,total_customer_value,avg_order_value,first_order_date_dt,last_order_date_dt,customer_life,recency,churn,cum_value,cum_ratio,customer_ratio
15225,32ba9eb2-7d94-11ea-80af-000d3a38a36f,Mobile,Offline,2020-04-18,2020-09-19,2020-04-30,2020-09-19,2.0,2.0,190.98,...,510.96,127.740000,2020-04-18,2020-09-19,154,254,True,11984478.26,0.799841,0.535372
6042,6493e79a-3d93-11ea-9e19-000d3a38a36f,Android App,Android App,2019-11-19,2020-06-07,2020-06-07,2019-11-19,2.0,1.0,191.97,...,510.95,170.316667,2019-11-19,2020-06-07,201,358,True,11984989.21,0.799876,0.535422
10881,3b6a6aa6-ab5d-11e9-a2fc-000d3a38a36f,Desktop,Offline,2017-11-19,2021-03-06,2017-11-19,2021-03-06,1.0,2.0,430.96,...,510.94,170.313333,2017-11-19,2021-03-06,1203,86,False,11985500.15,0.799910,0.535473
15365,ceee4e48-c54a-11ea-9dde-000d3a38a36f,Android App,Android App,2020-07-12,2021-02-12,2021-02-12,2021-02-02,1.0,2.0,420.95,...,510.94,170.313333,2020-07-12,2021-02-12,215,108,True,11986011.09,0.799944,0.535523
6467,0a91ff6a-a7b7-11e9-a2fc-000d3a38a36f,Desktop,Desktop,2019-04-13,2020-12-21,2020-12-21,2019-04-13,3.0,1.0,99.99,...,510.93,127.732500,2019-04-13,2020-12-21,618,161,True,11986522.02,0.799978,0.535573


## Genel Değerlendirme ve Öneriler
###  Temel Bulgular



Müşterilerin ortalama sipariş sayısı yaklaşık 5 olup genel müşteri kitlesi orta seviyede alışveriş sıklığına sahiptir.  
Gelir dağılımı incelendiğinde bazı müşterilerin yüksek harcama yaptığı ve gelir dağılımının dengesiz olduğu gözlemlenmiştir.  
Kanal bazlı analizde Android uygulamasının toplam gelirde lider olduğu, ancak iOS kullanıcılarının müşteri başına daha yüksek harcama yaptığı görülmüştür.

RFM ve Segment Analizi

Champions ve Loyal Customers segmentleri en değerli müşteri grubunu oluşturmaktadır.  
Cant Lose segmenti yüksek harcama potansiyeline sahip olmasına rağmen uzun süredir alışveriş yapmayan kritik müşterileri temsil etmektedir.  
Hibernating segmentinin en büyük grup olması müşteri kitlesinin önemli bir kısmının pasif olduğunu göstermektedir.

Churn Analizi

Churn olan müşteri sayısının aktif müşterilerden fazla olması müşteri kaybının önemli bir problem olduğunu göstermektedir.  
Aktif müşterilerin daha yüksek harcama, daha fazla sipariş ve daha uzun müşteri yaşam süresine sahip olduğu görülmüştür.  
Bu durum müşteri sadakatinin doğrudan gelir ile ilişkili olduğunu ortaya koymaktadır.

Pareto Analizi

Toplam gelirin yüzde 80’i müşteri kitlesinin yaklaşık yüzde 53’ü tarafından oluşturulmaktadır.  
Bu sonuç gelir dağılımının belirli bir müşteri grubunda yoğunlaştığını ancak klasik 80 20 kuralına tam olarak uymadığını göstermektedir.  
Gelirin geniş bir müşteri kitlesine yayılmış olması farklı segmentler için farklı stratejiler geliştirilmesi gerektiğini ortaya koymaktadır.

İş Önerileri

Pasif müşteriler için yeniden kazanım kampanyaları geliştirilmelidir.  
Cant Lose segmentine özel promosyonlar ve kişiselleştirilmiş teklifler sunulmalıdır.  
Champions ve Loyal Customers segmentleri için sadakat programları oluşturulmalıdır.  
iOS kullanıcılarının yüksek harcama potansiyeli göz önünde bulundurularak premium ürün ve kampanyalar geliştirilebilir.  
Android platformunda geniş müşteri kitlesi korunmalı ve kullanıcı sayısı artırılmaya devam edilmelidir.

Sonuç

Bu analizler sonucunda müşteri davranışları detaylı şekilde incelenmiş yüksek değerli müşteriler belirlenmiş ve müşteri kaybını azaltmaya yönelik stratejik öneriler geliştirilmiştir. Proje ERP sistemlerinde kullanılabilecek müşteri analitiği yaklaşımını yansıtmaktadır.